<a href="https://colab.research.google.com/github/DiFedorchuk/ML_Course/blob/main/HW_RecSys_Goodbooks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [1]:
from google.colab import files; files.upload('kaggle.json')   # оберіть kaggle.json

Saving kaggle.json to kaggle.json/kaggle.json


{'kaggle.json/kaggle.json': b'{\r\n"username": "dmytrofedorchuk92",\r\n"key": "KGAT_fd763d8047227792df6fca64181c3c3f"\r\n}'}

In [2]:
import os, shutil

# Create the parent directory for Kaggle config
os.makedirs("/root/.kaggle", exist_ok=True)

# Define the destination path
destination_file_path = "/root/.kaggle/kaggle.json"

# Remove existing destination if it's a directory
if os.path.isdir(destination_file_path):
    print(f"Warning: Removing existing directory at {destination_file_path} to resolve conflict.")
    shutil.rmtree(destination_file_path)
# If it's a file, remove it too to ensure a clean move
elif os.path.isfile(destination_file_path):
    print(f"Warning: Removing existing file at {destination_file_path} to resolve conflict.")
    os.remove(destination_file_path)

# Find the actual uploaded kaggle.json file.
# This loop handles cases where `files.upload()` might rename the file
# or put it into an unexpected directory structure (like 'kaggle.json/kaggle (1).json')
uploaded_kaggle_json_path = None
for root, _, files in os.walk('.'):
    for f in files:
        if f.startswith('kaggle') and f.endswith('.json'):
            uploaded_kaggle_json_path = os.path.join(root, f)
            break
    if uploaded_kaggle_json_path:
        break

if uploaded_kaggle_json_path:
    print(f"Moving {uploaded_kaggle_json_path} to {destination_file_path}")
    shutil.move(uploaded_kaggle_json_path, destination_file_path)
    os.chmod(destination_file_path, 0o600)
    print("Kaggle API key moved and permissions set successfully.")
else:
    print("Error: kaggle.json file not found in the current directory or its subdirectories. Please ensure you have uploaded it correctly.")

# Download dataset
!kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .


Moving ./kaggle.json/kaggle.json to /root/.kaggle/kaggle.json
Kaggle API key moved and permissions set successfully.
Dataset URL: https://www.kaggle.com/datasets/zygmunt/goodbooks-10k
License(s): CC-BY-SA-4.0
100% 11.6M/11.6M [00:00<00:00, 107MB/s] 



In [3]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings: (981756, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,2767052,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,41865,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,2657,Harper Lee,To Kill a Mockingbird,4.25
4,4671,F. Scott Fitzgerald,The Great Gatsby,3.89


In [4]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url'],
      dtype='object')

In [5]:
book_tags.columns

Index(['goodreads_book_id', 'tag_id', 'count'], dtype='object')

## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [6]:
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["id"], books["book_id"])) # Changed 'goodreads_book_id' to 'id'
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 811 / 10000

Розподіл жанрів:
genre
classics           455
contemporary       447
fantasy            283
historical         282
romance            232
young-adult        224
mystery            223
nonfiction         214
thriller           174
science-fiction    156
crime              139
horror              78
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1.0,0,0,0,0,0,0,0,0,1,0,1,0
2.0,0,0,0,0,0,1,0,0,1,0,0,0
3.0,1,1,1,0,0,0,0,1,0,0,0,0
6.0,0,0,0,0,0,0,0,0,1,0,1,0
27.0,0,0,0,0,0,1,0,0,1,0,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [7]:
TOP_BOOKS = 1500       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 793 | користувачів: 600 | книг: 14
Щільність: 0.0944


In [8]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 392 | користувачів з val-лайками: 100


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [10]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [11]:
# 1
item_emb = torch.nn.functional.normalize(item_feats, p=2, dim=1)
print("Shape:", item_emb.shape)
print(item_emb[:5])

Shape: torch.Size([14, 12])
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.7071, 0.0000, 0.0000, 0.7071,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4472, 0.4472, 0.0000, 0.4472, 0.0000, 0.0000, 0.0000,
         0.4472, 0.0000, 0.4472],
        [0.4472, 0.0000, 0.4472, 0.0000, 0.0000, 0.0000, 0.0000, 0.4472, 0.0000,
         0.4472, 0.4472, 0.0000],
        [0.4472, 0.0000, 0.4472, 0.4472, 0.4472, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.4472, 0.0000],
        [0.0000, 0.4472, 0.0000, 0.4472, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.4472, 0.4472, 0.4472]])


In [12]:
# 2
def user_vector(user_idx):
    liked_books_df = train_pos[train_pos["user_id"] == users[user_idx]]
    if liked_books_df.empty:
        return torch.zeros(n_genres)

    liked_item_indx = torch.tensor([item_to_idx[b] for b in liked_books_df["book_id"]])
    liked_item_ratings = torch.tensor(liked_books_df["rating"].values, dtype=torch.float32)
    embeddings_of_liked_items = item_emb[liked_item_indx]
    weighted_sum = torch.sum(embeddings_of_liked_items * liked_item_ratings.unsqueeze(1), dim=0)
    user_vec = torch.nn.functional.normalize(weighted_sum, p=2, dim=0)
    return user_vec

sample_user_idx = 0 # first user
sample_user_vec = user_vector(sample_user_idx)
print(f"User {users[sample_user_idx]} vector shape: {sample_user_vec.shape}")
print(f"User {users[sample_user_idx]} vector (first 5 elements): {sample_user_vec[:5]}")

User 75 vector shape: torch.Size([12])
User 75 vector (first 5 elements): tensor([0., 0., 0., 0., 0.])


In [13]:
# 3
def vsm_scores(user_idxs):
    user_vectors_list = []
    for u_idx in user_idxs:
        user_vectors_list.append(user_vector(u_idx))
    user_vectors = torch.stack(user_vectors_list) # (len(user_idxs), n_genres
    scores = user_vectors @ item_emb.T  # (len(user_idxs), n_genres) @ (n_genres, M) -> (len(user_idxs), M)
    return scores

print(f"Recall@10 for VSM: {recall_at_k(vsm_scores, k=10):.4f}")

# Top-5 recommendations
print("\nTop-5 recommendations:")
sample_user_id = users[sample_user_idx]
sample_user_scores = vsm_scores(torch.tensor([sample_user_idx]))[0]
scores_for_recommendation = sample_user_scores.clone()
for i in seen_by_user[sample_user_idx]:
    scores_for_recommendation[i] = -1e9

top_5_indices = torch.topk(scores_for_recommendation, 5).indices.tolist()
top_5_book_ids = [items[idx] for idx in top_5_indices]

for i, book_id in enumerate(top_5_book_ids):
    print(f"{i+1}. {title_of.get(book_id, 'Unknown Title')} (Book ID: {book_id})")

Recall@10 for VSM: 0.7238

Top-5 recommendations:
1. Everyday Italian: 125 Simple and Delicious Recipes (Book ID: 1192)
2. Neither Here nor There: Travels in Europe (Book ID: 27)
3. Memoirs of a Geisha (Book ID: 930)
4. The Broker (Book ID: 1110)
5. Pompeii (Book ID: 880)


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


Можливо це залежить від данних, а саме присутній шум, так як жанри беруться з тегів користувачів. Також лайки користувача можуть не повністю відображати його вподобання. Модель VSM не враховує складних ознак, лише середнє згачення ознак обєктів

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [17]:
# 1
import torch.nn.functional as F

class TwoTower(nn.Module):
    def __init__(self, num_users, item_features, embedding_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        n_genres = item_features.shape[1]
        self.item_projection = nn.Linear(n_genres, embedding_dim)
        # Register item_features as a buffer so it moves with the model to device
        self.register_buffer('item_features_static', item_features)

    def user_tower(self, user_idxs):
        user_emb = self.user_embedding(user_idxs)
        return F.normalize(user_emb, p=2, dim=1)

    def item_tower(self):
        # Apply the linear projection to the static item features
        item_embs = self.item_projection(self.item_features_static)
        return F.normalize(item_embs, p=2, dim=1)

    def forward(self, user_idxs):
        user_embs = self.user_tower(user_idxs)
        item_embs = self.item_tower()
        scores = user_embs @ item_embs.T
        return scores

In [19]:
# 2
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

EMBEDDING_DIM = 64
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20
BATCH_SIZE = 256
NUM_NEG_SAMPLES = 4
TEMPERATURE = 10.0

num_users = len(users)
model = TwoTower(num_users, item_feats, EMBEDDING_DIM)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.BCEWithLogitsLoss()

train_pos_dataset = TensorDataset(pos_u, pos_i)
train_pos_dataloader = DataLoader(train_pos_dataset, batch_size=BATCH_SIZE, shuffle=True)

model.train()
print("TwoTower model training")
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for user_idxs_batch, pos_item_idxs_batch in tqdm(train_pos_dataloader, leave=False):
        optimizer.zero_grad()
        batch_neg_item_idxs = []
        for u_idx, p_i_idx in zip(user_idxs_batch.tolist(), pos_item_idxs_batch.tolist()):
            all_item_indices = set(range(M))
            user_seen_items = seen_by_user[u_idx]
            candidate_neg_items = list(all_item_indices - user_seen_items)
            if len(candidate_neg_items) < NUM_NEG_SAMPLES:
                neg_items = torch.randint(0, M, (NUM_NEG_SAMPLES,)).tolist()
            else:
                neg_items = rng.choice(candidate_neg_items, NUM_NEG_SAMPLES, replace=False).tolist()
            batch_neg_item_idxs.extend(neg_items)

        neg_item_idxs_batch = torch.tensor(batch_neg_item_idxs)
        all_user_idxs_for_loss = user_idxs_batch.repeat_interleave(NUM_NEG_SAMPLES + 1)
        all_item_idxs_for_loss = torch.cat([pos_item_idxs_batch, neg_item_idxs_batch])

        labels = torch.cat([
            torch.ones(pos_item_idxs_batch.shape[0]),
            torch.zeros(neg_item_idxs_batch.shape[0])
        ])

        # Get embeddings from towers
        user_embs_for_loss = model.user_tower(all_user_idxs_for_loss)
        all_item_embeddings = model.item_tower()
        item_embs_for_loss = all_item_embeddings[all_item_idxs_for_loss]
        # Compute scores
        scores = torch.sum(user_embs_for_loss * item_embs_for_loss, dim=1)
        scores = scores * TEMPERATURE
        # Calculate loss
        loss = loss_fn(scores, labels)
        # Backpropagation
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {total_loss / len(train_pos_dataloader):.4f}")

print("Training complete.")



TwoTower model training


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1/20, Loss: 0.8548


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 2/20, Loss: 0.8442


...

Epoch 18/20, Loss: 0.7788


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 19/20, Loss: 0.7703


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 20/20, Loss: 0.7890
Training complete.


In [20]:
# 3
model.eval()
all_item_embeddings_trained = model.item_tower() # (M, EMBEDDING_DIM)

def tt_score_fn(user_idxs_tensor):
    user_embs = model.user_tower(user_idxs_tensor) # (num_users_in_batch, EMBEDDING_DIM)
    # Calculate scores
    scores = user_embs @ all_item_embeddings_trained.T # (num_users_in_batch, M)
    return scores * TEMPERATURE

print(f"\nRecall@10 for TwoTower: {recall_at_k(tt_score_fn, k=10):.4f}")

# top-5 recommendations for one user
sample_user_id = users[sample_user_idx]
sample_user_scores = tt_score_fn(torch.tensor([sample_user_idx]))[0] # Output is (M,) for a single user
scores_for_recommendation = sample_user_scores.clone()
for i in seen_by_user[sample_user_idx]:
    scores_for_recommendation[i] = -1e9 # Assign low score to seen items

top_5_indices = torch.topk(scores_for_recommendation, 5).indices.tolist()
top_5_book_ids = [items[idx] for idx in top_5_indices]

print(f"\nTop-5 recommendations from TwoTower for user {sample_user_id}:")
for i, book_id in enumerate(top_5_book_ids):
    print(f"{i+1}. {title_of.get(book_id, 'Unknown Title')} (Book ID: {book_id})")


Recall@10 for TwoTower: 0.6762

Top-5 recommendations from TwoTower for user 75:
1. The Diamond Age: or, A Young Lady's Illustrated Primer (Book ID: 827)
2. The Confusion (The Baroque Cycle, #2) (Book ID: 822)
3. Neither Here nor There: Travels in Europe (Book ID: 27)
4. Gravity's Rainbow (Book ID: 415)
5. Memoirs of a Geisha (Book ID: 930)


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [21]:
# 1
class NCF(nn.Module):
    def __init__(self, num_users, item_features, embedding_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        n_genres = item_features.shape[1]
        self.register_buffer('item_features_static', item_features)

        # MLP layers
        input_size = embedding_dim + n_genres
        self.mlp = nn.Sequential(
            nn.Linear(input_size, embedding_dim * 2), # Increased hidden_dim
            nn.ReLU(),
            nn.Linear(embedding_dim * 2, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, 1) # Output a single logit
        )

    def forward(self, user_idxs, item_idxs):
        user_emb = self.user_embedding(user_idxs)
        item_feats = self.item_features_static[item_idxs]
        # Concatenate
        concat_features = torch.cat((user_emb, item_feats), dim=1)
        # Pass through MLP
        logit = self.mlp(concat_features)
        return logit.squeeze(1)

In [22]:
# 2
num_users = len(users)
ncf_model = NCF(num_users, item_feats, EMBEDDING_DIM)

ncf_optimizer = torch.optim.Adam(ncf_model.parameters(), lr=LEARNING_RATE)
ncf_loss_fn = nn.BCEWithLogitsLoss()

train_pos_dataset = TensorDataset(pos_u, pos_i)
train_pos_dataloader = DataLoader(train_pos_dataset, batch_size=BATCH_SIZE, shuffle=True)

ncf_model.train()
print("NCF model training")
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    for user_idxs_batch, pos_item_idxs_batch in tqdm(train_pos_dataloader, leave=False):
        ncf_optimizer.zero_grad()

        # Generate negative samples for each user in the batch
        batch_neg_item_idxs = []
        for u_idx, p_i_idx in zip(user_idxs_batch.tolist(), pos_item_idxs_batch.tolist()):
            # Pool of items that have not been seen by the current user
            all_item_indices = set(range(M))
            user_seen_items = seen_by_user[u_idx]
            candidate_neg_items = list(all_item_indices - user_seen_items)

            if len(candidate_neg_items) < NUM_NEG_SAMPLES:
                # Fallback if not enough unique negative items are available
                neg_items = torch.randint(0, M, (NUM_NEG_SAMPLES,)).tolist()
            else:
                neg_items = rng.choice(candidate_neg_items, NUM_NEG_SAMPLES, replace=False).tolist()
            batch_neg_item_idxs.extend(neg_items)

        neg_item_idxs_batch = torch.tensor(batch_neg_item_idxs)

        # Combine positive and negative samples for loss calculation
        all_user_idxs_for_loss = user_idxs_batch.repeat_interleave(NUM_NEG_SAMPLES + 1)
        all_item_idxs_for_loss = torch.cat([pos_item_idxs_batch, neg_item_idxs_batch])

        labels = torch.cat([
            torch.ones(pos_item_idxs_batch.shape[0]),
            torch.zeros(neg_item_idxs_batch.shape[0])
        ])

        # Get scores from NCF model
        scores = ncf_model(all_user_idxs_for_loss, all_item_idxs_for_loss)

        # Calculate loss
        loss = ncf_loss_fn(scores, labels)

        # Backpropagation
        loss.backward()
        ncf_optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {total_loss / len(train_pos_dataloader):.4f}")

print("NCF Training complete.")

NCF model training


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1/20, Loss: 0.6807


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 2/20, Loss: 0.6457


...

Epoch 18/20, Loss: 0.5011


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 19/20, Loss: 0.4938


  0%|          | 0/2 [00:00<?, ?it/s]

Epoch 20/20, Loss: 0.4945
NCF Training complete.


In [23]:
# 3
def rank_ncf(user_idx, candidate_idxs):
    ncf_model.eval()
    with torch.no_grad():
        user_idxs_tensor = torch.full((len(candidate_idxs),), user_idx, dtype=torch.long)
        candidate_idxs_tensor = torch.tensor(candidate_idxs, dtype=torch.long)
        logits = ncf_model(user_idxs_tensor, candidate_idxs_tensor)
        scores = torch.sigmoid(logits)
    return scores

print("rank_ncf function defined.")

rank_ncf function defined.


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [24]:
def retrieve(user_idx, n_candidates):
    model.eval()
    with torch.no_grad():
        user_emb = model.user_tower(torch.tensor([user_idx])) # (1, EMBEDDING_DIM)
        item_embs = model.item_tower() # (M, EMBEDDING_DIM)
        # Calculate scores
        scores = (user_emb @ item_embs.T).squeeze(0) * TEMPERATURE # (M,)
        # Filter out seen items
        scores_for_selection = scores.clone()
        for i in seen_by_user[user_idx]:
            scores_for_selection[i] = -1e9 # Assign  low score to seen items
        # Get top n_candidates
        top_candidate_indices = torch.topk(scores_for_selection, n_candidates).indices.tolist()
    return top_candidate_indices

sample_user_idx_retrieval = 0
n_candidates_retrieval = 10
retrieved_candidates = retrieve(sample_user_idx_retrieval, n_candidates_retrieval)

print(f"Retrieved {n_candidates_retrieval} candidates for user {users[sample_user_idx_retrieval]}:")
for i, item_idx in enumerate(retrieved_candidates):
    book_id = items[item_idx]
    print(f"{i+1}. {title_of.get(book_id, 'Unknown Title')} (Book ID: {book_id})")

Retrieved 10 candidates for user 75:
1. The Diamond Age: or, A Young Lady's Illustrated Primer (Book ID: 827)
2. The Confusion (The Baroque Cycle, #2) (Book ID: 822)
3. Neither Here nor There: Travels in Europe (Book ID: 27)
4. Gravity's Rainbow (Book ID: 415)
5. Memoirs of a Geisha (Book ID: 930)
6. The Broker (Book ID: 1110)
7. Anthem (Book ID: 667)
8. Veronika Decides to Die (Book ID: 1431)
9. Everyday Italian: 125 Simple and Delicious Recipes (Book ID: 1192)
10. Digging to America (Book ID: 698)


**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Тому що NCF оцінює кожну пару, чим більше даних тим довше та затратніше буде проганяти кожну пару через модель, тому краще відібрати кандидатів з датасету та застосувати NFC

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


In [ ]:
1. Обмежений набір жанрів(12), тобто не враховуються інші аспекти такі як історичний період, складність тексту, стиль і тд.
Користувачі не взаємодіяли з усіма наявними книгами, тобто моделі складно передбачити через брак даних.
2. Покращити якість можна додавши:
Автора - можна групувати книги за автором, спільний автор теоритично може дати спільний жанр;
Рік - за роком можна розрізняти жанр(класика, сучасна література);
Середній рейтинг - показник популярності книги, можемо використовувати у моделі;
набір тегів через TF IDF - замість 12 жанрів, можна використовувати всі доступні теги;
Мова - можна визначити книги якими мовами є найбільш популярними;
Середній рейтинг користувача - середнє значення всіх оцінок які він поставив;
3. Однотипні рекомендації є неефективними, покращити це можна додавши штраф за однотипну рекомендацію, обрати фіксовану рекомендацію за одним жанромб
переранжування вибірки для різноманіття рекомендацій
4. Одразу порекомендує VSM та Two Tower, так як вони використовують категорійні ознаки книги(жанр, автор і тд).
Не порекомендує одразу NCF, він використовує ембеднг користувача та контентні ознаки книги(id)
5. Якщо користувач клікнув це ще не означає що він зацікавився та подивився відео, може багато клікати та швидко закривати відео.
Тому час перегляду краще відображає інтерес та задоволекість користувача
